# Language Modelling

## 1.0 Objectives

* Create n-gram frequency distribution with NLTK `FreqDist` and `ConditionalFreqDist`
* Compute conditional probability
* Spelling correction

In [5]:
import nltk
nltk.download('punkt_tab')
nltk.download("brown")
from nltk.corpus import brown
from nltk import FreqDist, bigrams, ConditionalFreqDist, trigrams, ngrams
from nltk.metrics.distance import edit_distance
from nltk.tokenize import TreebankWordTokenizer, TreebankWordTokenizer, word_tokenize
import string
puncts = string.punctuation
from nltk.corpus.reader.tagged import TreebankWordDetokenizer

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!


## 1.1 Frequency Distribution

In [6]:
# This is an example text

text = "This is this a nice this class Class in Class class not too nice sleeping too much nice and Nice and not and talking plenty plenty"
print(text)

This is this a nice this class Class in Class class not too nice sleeping too much nice and Nice and not and talking plenty plenty


We should convert all the text to lower case for uniformity.

In [7]:
text = text.lower()
print(text)

this is this a nice this class class in class class not too nice sleeping too much nice and nice and not and talking plenty plenty


In [8]:
text_tokenized = word_tokenize(text)
print(text_tokenized)

['this', 'is', 'this', 'a', 'nice', 'this', 'class', 'class', 'in', 'class', 'class', 'not', 'too', 'nice', 'sleeping', 'too', 'much', 'nice', 'and', 'nice', 'and', 'not', 'and', 'talking', 'plenty', 'plenty']


We can use the tokenised text to obtain unigram frequency distribution.

In [9]:
unigram_freq_text = FreqDist(text_tokenized)
print(unigram_freq_text)

<FreqDist with 13 samples and 26 outcomes>


Note that for FreqDist above, we only passed the tokenised text.

For `ConditionalFreqDist()`, we will pass the tokens into `bigrams()` method before passing it into `ConditionalFreqDist()`.

We will use this concept to create trigram frequency distribution.

Let us display the unigram frequency distribution.

In [10]:
freqs = []

# TODO: Loop through the items in unigram_freq_text and append each (word, frequency) pair to the freqs list.
for w in unigram_freq_text.items():
    freqs.append(w)

print(freqs)
print(len(freqs))

[('this', 3), ('is', 1), ('a', 1), ('nice', 4), ('class', 4), ('in', 1), ('not', 2), ('too', 2), ('sleeping', 1), ('much', 1), ('and', 3), ('talking', 1), ('plenty', 2)]
13


We can also use list comprehension.

In [11]:
# TODO: Use a list comprehension with unigram_freq_text.items() to display all word-frequency pairs.
print([w for w in unigram_freq_text.items()])

[('this', 3), ('is', 1), ('a', 1), ('nice', 4), ('class', 4), ('in', 1), ('not', 2), ('too', 2), ('sleeping', 1), ('much', 1), ('and', 3), ('talking', 1), ('plenty', 2)]


Let us do the same for bigrams. This time, we need to call ConditionalFreqDist() on the bigram tokens.

In [12]:
bigram_freq = ConditionalFreqDist(bigrams(text_tokenized))

We can display the bigram frequencies as shown in the next cell:

In [13]:
bigram_freqs = [w for w in bigram_freq.items()]
bigram_freqs

[('this', FreqDist({'is': 1, 'a': 1, 'class': 1})),
 ('is', FreqDist({'this': 1})),
 ('a', FreqDist({'nice': 1})),
 ('nice', FreqDist({'and': 2, 'this': 1, 'sleeping': 1})),
 ('class', FreqDist({'class': 2, 'in': 1, 'not': 1})),
 ('in', FreqDist({'class': 1})),
 ('not', FreqDist({'too': 1, 'and': 1})),
 ('too', FreqDist({'nice': 1, 'much': 1})),
 ('sleeping', FreqDist({'too': 1})),
 ('much', FreqDist({'nice': 1})),
 ('and', FreqDist({'nice': 1, 'not': 1, 'talking': 1})),
 ('talking', FreqDist({'plenty': 1})),
 ('plenty', FreqDist({'plenty': 1}))]

## 1.2 Frequency Distribution using Brown Corpus

We can now use what we learnt from the example text and apply it to get the unigram and bigram frequencies for the brown corpus.

We will first apply lower case to the tokens before passing them into FreqDist() and ConditionalFreqDist().

In [14]:
# TODO: Create a frequency distribution of all words in the Brown corpus (in lowercase)
words = FreqDist(w.lower() for w in brown.words())

# TODO: Create a conditional frequency distribution of all bigrams in the Brown corpus (in lowercase)
words_2g = ConditionalFreqDist(bigrams(w.lower() for w in brown.words()))

In [15]:
# Display one item from the unigram frequency distribution
for w in words.items():
    print(w)
    break

('the', 69971)


In [16]:
# Display one item from the bigram frequency distribution
for w in words_2g.items():
    print(w)
    break

('the', FreqDist({'first': 662, 'same': 628, 'most': 417, 'other': 416, '``': 405, 'new': 397, 'united': 393, 'world': 361, 'state': 271, 'two': 268, ...}))


Let us get the bigram frequency of "bellboy".

In [17]:
words_2g["bellboy"]

FreqDist({'to': 1, 'and': 1, 'across': 1, 'unlocked': 1})

In [18]:
words_2g["bellboy"].total() # Total frequencies

4

In [19]:
words_2g["bellboy"].N() # Another way to get total frequencies

4

How many times did the word "to" come after the word "bellboy"?

In [20]:
words_2g["bellboy"]["to"]

1

How many times did the word "unlocked" come after the word "bellboy"?

In [21]:
words_2g["bellboy"]["unlocked"]

1

$$p({word0, word1}) = p({word1} | {word0})$$
For instance, $$p({'bellboy', 'unlocked'}) = p({'unlocked'} | {'bellboy'})$$

In the following cell:

* the function `cond_prob() takes `['word0'], 'word1` and `ConditionalFreqDist` object as input and returns conditional probability of bigrams.
* the function `candidates()`
    * takes a word,
    * compute the edit distance between the word and every word in the unigram frequency distribution,
    * gets the minimum edit distance value and
    * returns the word with the minimum edit distance value.

In [22]:
def cond_prob_bigram(prev_word, word):
    freq_2g = words_2g[prev_word][word] # Get bigram counts for each bigram
    total_2g = words_2g[prev_word].N() # Get total of bigram counts
    return freq_2g / total_2g

def candidates(w):
    dists = [(word, edit_distance(w, word)) for word in words] # List of word and edit distance values for every word in words
    min_dist = min(x[1] for x in dists) # Get the minimum edit distance value
    return [y[0] for y in dists if y[1] == min_dist] # Get the word that is associated with the minimum edit distance value

In [23]:
prob_example = cond_prob_bigram('the', 'book')
print(f'The probability of "book" following "the, p("the", "book"): {prob_example}')

The probability of "book" following "the, p("the", "book"): 0.0007717482957225136


## 1.3 Exercise 1

Using `ConditionalFreqDist()`, create trigram frequency distribution with `brown.words()` to calculate trigram probabilities.

Note: Passing the tokens into trigrams() directly will not work.

HINT: https://stackoverflow.com/questions/38068539/finding-conditional-probability-of-trigram-in-python-nltk




In [24]:
tokens = [w.lower() for w in brown.words()]
trigram_freq = trigrams(tokens)
words_3g = ConditionalFreqDist(((w0, w1), w2) for w0, w1, w2 in trigram_freq)

## 1.4 Exercise 2
Consider the following:
$$
p(\text{that, day, was}) = p\bigg({was} | {{the, day}}\bigg)
$$

Compute the trigram conditional probability of `was` given the words `the day`.


In [25]:
def cond_prob_trigram(prev_word, word):
  return words_3g[prev_word][word] / words_3g[prev_word].N()

In [26]:
cond_prob_trigram(("the", "day"), "was")

0.024390243902439025

## 1.5 Exercise 3

Consider the sequence with a misspelt word in the middle as shown below:

```python
correct_middle("is","thjs","correct")

```

In [27]:
def correct_middle(prev_word, word, next_word):
    """Returns the correct word for a wrongly spelled word in the middle of a sequence
    """
    if word not in words:
        return max((w for w in candidates(word)), key=lambda w: cond_prob_bigram(prev_word, w) * cond_prob_bigram(w, next_word))
    else:
        return word

**Question**: Observe the function below that is used for spelling correction and try to think of where it would fail.

**Answer:** The `correct_middle()` function would not work if incorrectly spelled word is at the first or last position.

## 1.6 Exercise 4

Given a sequence of words: `prev_word`, `word`, `next_word`

Write `correct_first()` and `correct_last()` functions using the `correct_middle()` example above, using the parameters `prev_word` or `next_word` appropriately where the misspelt `word` is either the first or last token.


In [28]:
def correct_first(word, next_word):
    if word not in words:
        return max((w for w in candidates(word)), key=lambda w: cond_prob_bigram(w, next_word))
    else:
        return word

def correct_last(prev_word, word):
    if word not in words:
        return max((w for w in candidates(word)), key=lambda w: cond_prob_bigram(prev_word, w))
    else:
        return word

In [29]:
correct_last('the', 'rivr')

'river'

## 1.7 Exercise 5
Some lists are given below. Each sentence of a list contains a tuple of text with an incorrect word and the incorrect/misspelt word.

Using `correct_first()`, `correct_last()` and `correct_middle()`, write a function that
* takes the list of tuples and
* returns the correct spelling.

In [30]:
# Solution
def apply_correction(input_list):
    """
    1. Iterate through all_incorrect_lists variable
    2. Get tuple with sentence, misspelled word
    2.1 tokenize the sentence
    3. Index of misspelled word
    4. Select the right condition based on the index to apply the correct one to
        correct_first, corrrect_last, correct_middle
    5. get the corrected word
    6. replace in the tokenized sentence using index
    7. join/detokenize
    8. append to correct_list
    """

    correct_list = []
    word_tokenizer = TreebankWordTokenizer()
    word_detokenizer = TreebankWordDetokenizer()

    for sentence, incorrect_word in input_list:
        tok_sentence = word_tokenizer.tokenize(sentence)
        if len(tok_sentence)==1:
            correct_list.append(word_detokenizer.detokenize(tok_sentence))
            continue
        target_index = tok_sentence.index(incorrect_word)

        correct_word = None
        if target_index == 0:
            next_word = tok_sentence[target_index+1]
            correct_word = correct_first(incorrect_word, next_word)
        elif target_index == len(tok_sentence) - 1:
            prev_word = tok_sentence[target_index-1]
            correct_word = correct_last(prev_word, incorrect_word)
        elif target_index < len(tok_sentence) and target_index>0:
            prev_word = tok_sentence[target_index-1]
            next_word= tok_sentence[target_index+1]
            correct_word = correct_middle(prev_word, incorrect_word, next_word)
        if correct_word is None:
            print(f'Error in: {sentence}')
        tok_sentence[target_index] = correct_word
        correct_list.append(word_detokenizer.detokenize(tok_sentence))
    return correct_list


In [31]:
incorrect_list1 = [('is thjs correct', 'thjs'),
 ('there is batttle lines drawn over there', 'batttle'),
 ("nobody is right if everyody is wrong", "everyody"),
 ("young people spaking their minds", "spaking"),
 ("getting so much resiztance from behind", "resiztance")]

incorrect_list2 = [('togh day at work', 'togh'),
                   ('therere is a tree there', 'therere'),
                   ('wineee and cheese', 'wineee')]

incorrect_list3 = [
    ('where are my tings', 'tings'),
    ('the dog jumped in the rivr', 'rivr')
]

all_incorrect_lists = incorrect_list1 + incorrect_list2 + incorrect_list3

In [32]:
apply_correction(all_incorrect_lists)

['is this correct',
 'there is battle lines drawn over there',
 'nobody is right if everybody is wrong',
 'young people speaking their minds',
 'getting so much resistance from behind',
 'tough day at work',
 'there is a tree there',
 'dineen and cheese',
 'where are my things',
 'the dog jumped in the river']

## The End